In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical

: 

### 1. Cargar los datos

In [ ]:
df = pd.read_csv('data/Leukemia_GSE9476.csv')

# Separar características (genes) y etiquetas (tipo de leucemia)
X = df.drop(['type'], axis=1)
if 'samples' in X.columns:
    X = X.drop(['samples'], axis=1)
y_text = df['type']

### 2. Codificar las etiquetas a números (0, 1, 2, 3, 4)

In [ ]:
encoder = LabelEncoder()
y = encoder.fit_transform(y_text)
num_classes = len(encoder.classes_)
print(f"Clases detectadas: {encoder.classes_}")


### 3. Configurar Validación Cruzada (5-Fold)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_no = 1
accuracies = []

for train_index, test_index in skf.split(X, y):
    print(f"\n--- Entrenando Fold {fold_no} ---")
    
    # Separar datos de entrenamiento y prueba para este fold
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # 4. Normalización (Fit en train, transform en train y test)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 5. Feature Selection: Quedarnos con los 500 genes más informativos
    selector = SelectKBest(f_classif, k=500)
    X_train_selected = selector.fit_transform(X_train_scaled, y_train)
    X_test_selected = selector.transform(X_test_scaled)
    
    # 6. Balanceo con SMOTE (¡SOLO en los datos de entrenamiento!)
    smote = SMOTE(random_state=42)
    X_train_smote, y_train_smote = smote.fit_resample(X_train_selected, y_train)
    
    # Convertir etiquetas a formato categórico para Keras
    y_train_cat = to_categorical(y_train_smote, num_classes)
    y_test_cat = to_categorical(y_test, num_classes)
    
    # 7. Construir la arquitectura de la red neuronal (MLP)
    model = Sequential([
        Dense(128, activation='relu', input_shape=(500,), kernel_regularizer=l2(0.01)),
        Dropout(0.5),
        Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
        Dropout(0.3),
        Dense(num_classes, activation='softmax') # 5 salidas para las 5 clases
    ])
    
    model.compile(optimizer='adam', 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])
    
    # 8. Entrenamiento
    model.fit(X_train_smote, y_train_cat, 
              epochs=50, 
              batch_size=16, 
              verbose=0)
    
    # 9. Evaluamos en el conjunto de prueba de este fold
    scores = model.evaluate(X_test_selected, y_test_cat, verbose=0)
    print(f"Precisión en Fold {fold_no}: {scores[1]*100:.2f}%")
    accuracies.append(scores[1] * 100)
    
    fold_no += 1

# Resultado final

In [ ]:
print(f"\n--- RESULTADO FINAL ---")
print(f"Precisión Media: {np.mean(accuracies):.2f}% (+/- {np.std(accuracies):.2f}%)")

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px

def guardar_modelo_entrenado(modelo, nombre_archivo='../models/modelo_leucemia.keras'):
    try:
        modelo.save(nombre_archivo)
        print(f"modelo guardado en: {nombre_archivo}")
    except Exception as e:
        print(f"Error al guardar el modelo: {e}")

def exportar_biomarcadores(selector, nombres_caracteristicas, archivo_salida='../results/biomarcadores.csv', top_k=20):
    """
    extrae y formatea los genes con mayor significancia estadística.
    """
    try:
        puntuaciones = selector.scores_
        df_importancia = pd.DataFrame({
            'gen_id': nombres_caracteristicas,
            'f_score': puntuaciones
        })

        top_genes = df_importancia.sort_values(by='f_score', ascending=False).head(top_k)

        print("-" * 50)
        print(f"Análisis de biomarcadores (top {top_k})")
        print("-" * 50)
        print(top_genes.to_string(index=False))
        print("-" * 50)

        top_genes.to_csv(archivo_salida, index=False)
        return top_genes

    except Exception as e:
        print(f"Error durante la extracción de datos: {e}")
        return None

def generar_reporte_plotly(datos, archivo_html='../plots/reporte_interactivo.html'):
    try:
        datos_ordenados = datos.sort_values(by='f_score', ascending=True)
        
        fig = px.bar(
            datos_ordenados, 
            x='f_score', 
            y='gen_id', 
            orientation='h',
            title='Ranking de genes por importancia diagnóstica',
            labels={'f_score': 'relevancia estadística (f-score)', 'gen_id': 'identificador de gen'}
        )
        
        fig.update_traces(marker_color='#2c3e50')
        fig.update_layout(
            template='plotly_white',
            title_font=dict(size=18),
            xaxis_title_font=dict(size=14),
            yaxis_title_font=dict(size=14)
        )
        
        fig.write_html(archivo_html)
        print(f"Reporte visual interactivo generado en: {archivo_html}")
        
    except Exception as e:
        print(f"Error al generar el reporte interactivo: {e}")

if __name__ == "__main__":
    datos_genes = exportar_biomarcadores(selector, X.columns)
    if datos_genes is not None:
        generar_reporte_plotly(datos_genes)
    guardar_modelo_entrenado(model)